### Self Attention with Trainable Weights
3 trainable weight matrices are need: query, key, and value.

In [26]:
import torch

In [27]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

We will implement the self-attention mechanism step by step by introducing the three trainable weight matrices Wq, Wk, and Wv. These three matrices are used to project the embedded input tokens, x(i), into query, key, and value vectors, respectively

Start by showing the calculations for input 2:

In [28]:
x_2 = inputs[1]
d_in = inputs.shape[1]  # the input embedding size (3)
d_out = 2               # the output embedding size for demonstration, for gpt it is the same as the input size

In [29]:
torch.manual_seed(123)

# requires_grad=False to reduce clutter, but needs to be True to update for training
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

print(x_2)
print(W_query)

tensor([0.5500, 0.8700, 0.6600])
Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])


Compute the vectors:

In [30]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

tensor([0.4306, 1.4551])


#### Obtain the vectors for all inputs

In [31]:
queries = inputs @ W_query
keys = inputs @ W_key
values = inputs @ W_value
print(keys.shape)

torch.Size([6, 2])


#### Compute the attention score for input 2 to itself
The dot product between query and key vectors:

In [32]:
key_2 = keys[1]
attn_score_22 = query_2.dot(key_2)
print(f"query_2:\n{query_2}\n")
print(f"key_2:\n{key_2}\n")
print(f"attn_score_22:\n{attn_score_22}")

query_2:
tensor([0.4306, 1.4551])

key_2:
tensor([0.4433, 1.1419])

attn_score_22:
1.8523844480514526


#### Generalize to compute all attention scores for input 2
So it is the dot product between the query input 2's query vector and the key vectors of the other inputs:

In [35]:
attn_scores_2 = query_2 @ keys.T
print(f"query_2:\n{query_2}\n")
print(f"keys:\n{keys}\n")
print(f"attn_score_2:\n{attn_scores_2}")

query_2:
tensor([0.4306, 1.4551])

keys:
tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]])

attn_score_2:
tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


#### Attention scores to weights
Scale the attention scores by dividing them by the square root of the embedding dimension and apply them to softmax.

In [40]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)  # the last dim: across columns
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


#### Compute the context vector

In [42]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


### A self-attention class

In [46]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [48]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


### A self-attention class using linear layers

In [55]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [57]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


nn.Linear uses a different weight initialization scheme, so the results are different.

### Assign the weights from v2 to v1 to verify the calculations are the same

In [75]:
print(f"sa_v1 W_query:\ntype: {type(sa_v1.W_query.data)}")
print(sa_v1.W_query)

print(f"\nsa_v2 W_query:\ntype: {type(sa_v2.W_query.weight.T)}")
print(sa_v2.W_query.weight.T)

sa_v1 W_query:
type: <class 'torch.Tensor'>
Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]], requires_grad=True)

sa_v2 W_query:
type: <class 'torch.Tensor'>
tensor([[ 0.3161, -0.1683],
        [ 0.4568, -0.3379],
        [ 0.5118, -0.0918]], grad_fn=<PermuteBackward0>)


In [77]:
sa_v1 = SelfAttention_v1(d_in, d_out)

sa_v1.W_query.data = sa_v2.W_query.weight.T
sa_v1.W_key.data = sa_v2.W_key.weight.T
sa_v1.W_value.data = sa_v2.W_value.weight.T

print(sa_v1(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
